# **Ejercicio 3 - Consultas directas sobre archivos Parquet**

En este ejercicio todas las consultas se hacen con DuckDB directamente sobre los archivos Parquet de `data/raw/`, sin importarlos previamente a una tabla.

In [1]:
import duckdb

ANIO = 2026  # este ejercicio documenta los datos de 2026

RUTA_YELLOW = f"../data/raw/yellow/{ANIO}/*.parquet"
RUTA_GREEN = f"../data/raw/green/{ANIO}/*.parquet"
RUTA_TODOS = f"../data/raw/*/{ANIO}/*.parquet"

con = duckdb.connect()  # base en memoria: los datos se leen desde los Parquet

## **3.1 Determine la cantidad de archivos disponibles**

Se tienen 16 archivos Parquet disponibles, 8 de taxis amarillos y 8 de taxis verdes, que corresponden a los meses de enero a agosto de 2026.

In [2]:
archivos = con.sql(f"""
    SELECT regexp_extract(file, '(yellow|green)_tripdata', 1) AS tipo,
           COUNT(*)                                          AS archivos
    FROM glob('{RUTA_TODOS}')
    GROUP BY tipo
    ORDER BY tipo DESC
""").df()

print(f"Archivos disponibles: {archivos['archivos'].sum()}")
archivos

Archivos disponibles: 16


,tipo,archivos
0,yellow,8
1,green,8


## **3.2 Determine la cantidad de registros disponibles**

Se tienen 30,040,469 registros disponibles, de los cuales 29,703,355 corresponden a taxis amarillos y 337,114 a taxis verdes; es decir, los taxis verdes representan apenas alrededor del 1.1 % de los viajes.

In [3]:
registros = con.sql(f"""
    SELECT regexp_extract(filename, '(yellow|green)_tripdata', 1) AS tipo,
           COUNT(*)                                              AS registros
    FROM read_parquet('{RUTA_TODOS}', filename = true, union_by_name = true)
    GROUP BY tipo
    ORDER BY tipo DESC
""").df()

print(f"Registros disponibles: {registros['registros'].sum():,}")
registros

Registros disponibles: 30,040,469


,tipo,registros
0,yellow,29703355
1,green,337114


## **3.3 Identifique las columnas presentes en los archivos**

Los taxis amarillos tienen 21 columnas y los verdes 22; comparten 18 con el mismo nombre, mientras que las columnas de fecha cambian de prefijo según el sistema que reporta el viaje (`tpep_` para amarillos y `lpep_` para verdes). Cabe mencionar que los archivos no tienen todos las mismas columnas: `request_source` aparece únicamente a partir de junio de 2026, por lo que se utiliza `union_by_name` para obtener el esquema combinado de todos los archivos y no solo el del primero. Las columnas con las que se trabaja son:

- **VendorID:** proveedor de la tecnología que registró el viaje.
- **tpep_pickup_datetime / lpep_pickup_datetime:** fecha y hora en que se activó el taxímetro, es decir, el inicio del viaje.
- **tpep_dropoff_datetime / lpep_dropoff_datetime:** fecha y hora en que se desactivó el taxímetro, es decir, el fin del viaje.
- **passenger_count:** cantidad de pasajeros, ingresada por el conductor.
- **trip_distance:** distancia del viaje en millas, reportada por el taxímetro.
- **RatecodeID:** tarifa aplicada al final del viaje (estándar, aeropuerto JFK, Newark, negociada, entre otras).
- **store_and_fwd_flag:** indica si el registro se guardó en el vehículo antes de enviarse por falta de conexión (`Y`/`N`).
- **PULocationID / DOLocationID:** zona de la TLC donde inicia y termina el viaje.
- **payment_type:** forma de pago (tarjeta, efectivo, sin cargo, disputa, entre otras).
- **fare_amount:** tarifa calculada por tiempo y distancia.
- **extra:** recargos adicionales, como el de hora pico o nocturno.
- **mta_tax:** impuesto de la MTA aplicado al viaje.
- **tip_amount:** propina; solo se registra para pagos con tarjeta, por lo que las propinas en efectivo no aparecen.
- **tolls_amount:** total de peajes pagados durante el viaje.
- **improvement_surcharge:** recargo fijo de mejora aplicado a cada viaje.
- **total_amount:** monto total cobrado al pasajero, sin incluir propinas en efectivo.
- **congestion_surcharge:** recargo por congestión del Estado de Nueva York.
- **cbd_congestion_fee:** cargo por viajes dentro de la zona de congestión de Manhattan.
- **Airport_fee** (solo amarillos): cargo por recoger pasajeros en los aeropuertos LaGuardia y JFK.
- **ehail_fee** (solo verdes): cargo por solicitar el viaje de forma electrónica.
- **trip_type** (solo verdes): indica si el viaje se tomó en la calle o fue despachado por una central.
- **request_source** (desde junio de 2026): origen de la solicitud del viaje; solo tiene valor en los viajes solicitados por aplicación, con códigos de licencia de la TLC como `HV0003` o `HV0005`.

In [4]:
import pandas as pd

columnas_yellow = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{RUTA_YELLOW}', union_by_name = true)").df()["column_name"].tolist()
columnas_green = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{RUTA_GREEN}', union_by_name = true)").df()["column_name"].tolist()

print(f"Columnas yellow: {len(columnas_yellow)}")
print(f"Columnas green : {len(columnas_green)}")
print(f"En común       : {len(set(columnas_yellow) & set(columnas_green))}")

todas = columnas_yellow + [c for c in columnas_green if c not in columnas_yellow]
pd.DataFrame({
    "columna": todas,
    "yellow": [c in columnas_yellow for c in todas],
    "green": [c in columnas_green for c in todas],
})

Columnas yellow: 21
Columnas green : 22
En común       : 18


,columna,yellow,green
0,VendorID,True,True
1,tpep_pickup_datetime,True,False
2,tpep_dropoff_datetime,True,False
3,passenger_count,True,True
4,trip_distance,True,True
5,RatecodeID,True,True
6,store_and_fwd_flag,True,True
7,PULocationID,True,True
8,DOLocationID,True,True
9,payment_type,True,True


## **3.4 Determine los tipos de datos de las columnas**

Las columnas de fecha vienen como `TIMESTAMP`, por lo que no es necesario convertirlas desde texto, y los montos y la distancia vienen como `DOUBLE`. Por otro lado, columnas categóricas como `payment_type`, `RatecodeID` o `trip_type` vienen como `BIGINT` (entero de 8 bytes), aunque solo toman unos cuantos valores; incluso hay una inconsistencia, ya que `VendorID` y las zonas vienen como `INTEGER` mientras que los demás códigos son `BIGINT`.

Cabe mencionar que en disco este costo casi no se nota, ya que Parquet comprime muy bien columnas con pocos valores distintos; por ejemplo, `payment_type` ocupa alrededor de 2.6 MB comprimido para los 30 millones de registros. El problema aparece al cargar todo en memoria, donde la misma columna como entero de 8 bytes ocuparía cerca de 229 MB. Es por esto que leer únicamente las columnas que necesita cada consulta, como lo hace DuckDB, evita cargar en RAM datos que no se van a utilizar.

In [5]:
tipos_yellow = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{RUTA_YELLOW}', union_by_name = true)").df()[["column_name", "column_type"]]
tipos_green = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{RUTA_GREEN}', union_by_name = true)").df()[["column_name", "column_type"]]

tipos = tipos_yellow.merge(tipos_green, on="column_name", how="outer", suffixes=("_yellow", "_green"))
tipos["orden"] = tipos["column_name"].map({c: i for i, c in enumerate(todas)})
tipos.sort_values("orden").drop(columns="orden").reset_index(drop=True)

,column_name,column_type_yellow,column_type_green
0,VendorID,INTEGER,INTEGER
1,tpep_pickup_datetime,TIMESTAMP,NaN
2,tpep_dropoff_datetime,TIMESTAMP,NaN
3,passenger_count,BIGINT,BIGINT
4,trip_distance,DOUBLE,DOUBLE
5,RatecodeID,BIGINT,BIGINT
6,store_and_fwd_flag,VARCHAR,VARCHAR
7,PULocationID,INTEGER,INTEGER
8,DOLocationID,INTEGER,INTEGER
9,payment_type,BIGINT,BIGINT


In [6]:
# Tamaño en disco (Parquet comprimido) vs tamaño estimado en memoria como entero de 8 bytes
total_registros = int(registros["registros"].sum())

con.sql(f"""
    SELECT path_in_schema                                     AS columna,
           ROUND(SUM(total_compressed_size) / 1024 ** 2, 1)   AS mb_en_disco,
           ROUND({total_registros} * 8 / 1024 ** 2, 1)        AS mb_en_memoria_bigint
    FROM parquet_metadata('{RUTA_TODOS}')
    WHERE path_in_schema IN ('payment_type', 'RatecodeID', 'passenger_count')
    GROUP BY path_in_schema
    ORDER BY path_in_schema
""").df()

,columna,mb_en_disco,mb_en_memoria_bigint
0,RatecodeID,2.2,229.2
1,passenger_count,3.6,229.2
2,payment_type,2.6,229.2


## **3.5 Obtenga una muestra de los registros**

Se obtiene una muestra reproducible de 5 registros por tipo de taxi utilizando la semilla 123. Para esto, los registros de cada tipo se ordenan a partir de un hash de su archivo, su número de fila y la semilla, y se toman los primeros 5; de esta forma, la muestra es siempre la misma sin importar cuántas veces se ejecute, y abarca distintos meses. Se descartó `USING SAMPLE reservoir`, ya que con semilla fija tomaba únicamente registros de enero, por lo que la muestra no era representativa de los demás meses.

In [7]:
SEMILLA = 123

def muestra_por_tipo(tipo: str, n: int = 5) -> str:
    """Toma n registros de un tipo de taxi, ordenados por un hash que depende de la semilla."""
    return f"""
        (SELECT '{tipo}' AS tipo, *
         FROM read_parquet('../data/raw/{tipo}/{ANIO}/*.parquet', filename = true, file_row_number = true)
         ORDER BY hash(filename, file_row_number, {SEMILLA})
         LIMIT {n})
    """

muestra = con.sql(f"""
    SELECT * EXCLUDE (filename, file_row_number)
    FROM ({muestra_por_tipo('yellow')} UNION ALL BY NAME {muestra_por_tipo('green')})
    ORDER BY tipo DESC, COALESCE(tpep_pickup_datetime, lpep_pickup_datetime)
""").df()
muestra

,tipo,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,lpep_pickup_datetime,lpep_dropoff_datetime,ehail_fee,trip_type
0,yellow,2,2026-02-07 19:49:07,2026-02-07 19:53:01,3,0.38,1,N,79,79,...,0.0,1.0,9.85,2.50,0.0,0.75,NaT,NaT,NaN,<NA>
1,yellow,2,2026-04-28 10:15:47,2026-04-28 10:29:38,1,1.02,1,N,164,161,...,0.0,1.0,21.06,2.50,0.0,0.75,NaT,NaT,NaN,<NA>
2,yellow,1,2026-07-07 05:32:39,2026-07-07 05:51:19,1,9.40,1,N,132,28,...,0.0,1.0,41.10,0.00,2.0,0.00,NaT,NaT,NaN,<NA>
3,yellow,1,2026-07-17 20:19:51,2026-07-17 20:30:20,<NA>,2.00,<NA>,NaN,143,236,...,0.0,1.0,30.20,NaN,NaN,0.00,NaT,NaT,NaN,<NA>
4,yellow,2,2026-07-24 20:21:23,2026-07-24 20:30:44,<NA>,3.27,<NA>,NaN,7,138,...,0.0,1.0,30.22,NaN,NaN,0.00,NaT,NaT,NaN,<NA>
5,green,2,NaT,NaT,1,0.84,1,N,179,223,...,0.0,1.0,9.40,0.00,NaN,0.00,2026-01-12 12:37:16,2026-01-12 12:44:17,NaN,1
6,green,1,NaT,NaT,0,2.00,1,N,74,236,...,0.0,1.0,22.95,2.75,NaN,0.00,2026-04-15 09:36:58,2026-04-15 09:51:13,NaN,1
7,green,2,NaT,NaT,5,15.21,1,N,260,201,...,0.0,1.0,72.00,0.00,NaN,0.00,2026-05-22 23:45:51,2026-05-23 00:48:15,NaN,1
8,green,2,NaT,NaT,1,4.23,1,N,116,43,...,0.0,1.0,27.24,0.00,NaN,0.00,2026-05-26 07:26:56,2026-05-26 07:44:28,NaN,1
9,green,2,NaT,NaT,1,3.41,1,N,75,163,...,0.0,1.0,36.96,2.75,NaN,0.75,2026-07-21 17:06:24,2026-07-21 17:30:12,NaN,1


## **3.6 Problemas de calidad de datos identificados**

- **Bloque de registros incompletos:** 7,716,688 registros de amarillos (26 %) y 48,775 de verdes (14.5 %) tienen nulos al mismo tiempo en `passenger_count`, `RatecodeID`, `store_and_fwd_flag` y `congestion_surcharge`. Además, en los amarillos estos registros tienen `payment_type = 0`, que no corresponde a una forma de pago tradicional (el diccionario de la TLC lo describe como *Flex Fare*), y en los verdes `payment_type` también es nulo.
- **Fechas fuera de rango:** 244 viajes tienen una fecha que no corresponde al mes de su archivo, y 31 ni siquiera son de 2026, con fechas como 2001 o 2008.
- **Duraciones inválidas:** 15 viajes terminan antes de empezar, 371,902 tienen duración de cero y 267 duran más de 24 horas.
- **Distancias inválidas:** 964,443 viajes tienen distancia de cero y 1,295 superan las 100 millas, con un máximo de más de 328 mil millas.
- **Montos negativos:** 158,363 viajes tienen tarifa negativa y 162,858 un total negativo, que podrían corresponder a reembolsos o ajustes.
- **Pasajeros en cero:** 95,886 viajes registran 0 pasajeros.
- **Códigos desconocidos:** 769,695 viajes tienen `RatecodeID = 99` (tarifa desconocida) y 207,372 inician o terminan en las zonas 264/265, que la TLC usa para ubicaciones desconocidas.
- **Columna sin información:** `ehail_fee` está vacía en el 100 % de los registros de verdes.
- **Esquema inconsistente entre tipos y entre meses:** las fechas cambian de nombre (`tpep_`/`lpep_`), hay columnas exclusivas de cada tipo, los códigos categóricos mezclan `INTEGER` y `BIGINT`, y la columna `request_source` solo existe a partir de junio de 2026.

Estos registros no se eliminan en este punto; se documentan para tomarlos en cuenta en el análisis del Ejercicio 4.

In [8]:
con.sql(f"""
    CREATE OR REPLACE VIEW viajes AS
    SELECT regexp_extract(filename, '(yellow|green)_tripdata', 1)         AS tipo,
           regexp_extract(filename, '_(\d{{4}}-\d{{2}})\.parquet', 1)    AS mes_archivo,
           COALESCE(tpep_pickup_datetime, lpep_pickup_datetime)            AS pickup,
           COALESCE(tpep_dropoff_datetime, lpep_dropoff_datetime)          AS dropoff,
           *
    FROM read_parquet('{RUTA_TODOS}', filename = true, union_by_name = true)
""")

problemas = con.sql("""
    SELECT tipo,
           COUNT(*)                                                                    AS total,
           COUNT(*) FILTER (WHERE passenger_count IS NULL AND RatecodeID IS NULL
                              AND store_and_fwd_flag IS NULL AND congestion_surcharge IS NULL)
                                                                                       AS bloque_incompleto,
           COUNT(*) FILTER (WHERE strftime(pickup, '%Y-%m') <> mes_archivo)            AS fuera_del_mes,
           COUNT(*) FILTER (WHERE year(pickup) <> 2026)                                AS fuera_de_2026,
           COUNT(*) FILTER (WHERE dropoff < pickup)                                    AS termina_antes_de_iniciar,
           COUNT(*) FILTER (WHERE dropoff = pickup)                                    AS duracion_cero,
           COUNT(*) FILTER (WHERE dropoff - pickup > INTERVAL 24 HOUR)                 AS duracion_mayor_24h,
           COUNT(*) FILTER (WHERE trip_distance = 0)                                   AS distancia_cero,
           COUNT(*) FILTER (WHERE trip_distance > 100)                                 AS distancia_mayor_100,
           COUNT(*) FILTER (WHERE fare_amount < 0)                                     AS tarifa_negativa,
           COUNT(*) FILTER (WHERE total_amount < 0)                                    AS total_negativo,
           COUNT(*) FILTER (WHERE passenger_count = 0)                                 AS pasajeros_cero,
           COUNT(*) FILTER (WHERE RatecodeID = 99)                                     AS ratecode_99,
           COUNT(*) FILTER (WHERE PULocationID IN (264, 265) OR DOLocationID IN (264, 265))
                                                                                       AS zona_desconocida,
           COUNT(*) FILTER (WHERE tipo = 'green' AND ehail_fee IS NOT NULL)            AS ehail_fee_con_valor
    FROM viajes
    GROUP BY tipo
    ORDER BY tipo DESC
""").df().set_index("tipo").T

problemas["total"] = problemas.sum(axis=1)
problemas["pct_yellow"] = (100 * problemas["yellow"] / problemas.loc["total", "yellow"]).round(3)
problemas["pct_green"] = (100 * problemas["green"] / problemas.loc["total", "green"]).round(3)
problemas

tipo,yellow,green,total,pct_yellow,pct_green
total,29703355,337114,30040469,100.000,100.000
bloque_incompleto,7716688,48775,7765463,25.979,14.468
fuera_del_mes,146,98,244,0.000,0.029
fuera_de_2026,17,14,31,0.000,0.004
termina_antes_de_iniciar,10,5,15,0.000,0.001
duracion_cero,371673,229,371902,1.251,0.068
duracion_mayor_24h,263,4,267,0.001,0.001
distancia_cero,952231,12212,964443,3.206,3.623
distancia_mayor_100,1223,72,1295,0.004,0.021
tarifa_negativa,157364,999,158363,0.530,0.296


In [9]:
# Evidencia del bloque incompleto: los nulos aparecen juntos y, en amarillos, con payment_type = 0
con.sql("""
    SELECT tipo,
           passenger_count IS NULL AS sin_datos,
           payment_type,
           COUNT(*)                AS registros
    FROM viajes
    WHERE passenger_count IS NULL OR payment_type = 0
    GROUP BY ALL
    ORDER BY tipo DESC, payment_type
""").df()

,tipo,sin_datos,payment_type,registros
0,yellow,True,0,7716688
1,green,True,<NA>,48775


In [10]:
# Evidencia de fechas y distancias extremas
con.sql("""
    SELECT tipo,
           MIN(pickup)        AS pickup_minimo,
           MAX(pickup)        AS pickup_maximo,
           MAX(trip_distance) AS distancia_maxima,
           MIN(total_amount)  AS total_minimo,
           MAX(total_amount)  AS total_maximo
    FROM viajes
    GROUP BY tipo
    ORDER BY tipo DESC
""").df()

,tipo,pickup_minimo,pickup_maximo,distancia_maxima,total_minimo,total_maximo
0,yellow,2001-01-01 09:23:58,2026-08-31 23:59:59,328522.20,-2560.2,7053.5
1,green,2008-12-31 17:35:31,2026-08-31 23:58:28,179830.92,-501.5,1678.2


## **3.7 Consultas utilizando directamente los archivos Parquet**

Todas las consultas de este ejercicio se realizaron directamente sobre los archivos Parquet de `data/raw/` mediante `glob`, `read_parquet` y `parquet_metadata`, sin importar los datos previamente a una tabla. La vista `viajes` del 3.6 tampoco copia datos, ya que solo guarda la consulta y vuelve a leer los Parquet cada vez que se utiliza.

## **3.8 Documentación de las consultas**

La siguiente tabla resume las consultas utilizadas en este ejercicio. El SQL de cada una se encuentra en la celda de código de su inciso y, de forma consolidada, en [`../sql/ejercicio_3.sql`](../sql/ejercicio_3.sql), identificadas de Q1 a Q10.

| Consulta | Inciso | Objetivo | Archivos fuente | Resultado | Decisión |
|---|---|---|---|---|---|
| Q1 | 3.1 | Contar los archivos disponibles por tipo de taxi | `data/raw/*/2026/*.parquet` (solo se listan) | 16 archivos: 8 amarillos y 8 verdes | Se confirma que se cuenta con enero a agosto de 2026 para ambos tipos |
| Q2 | 3.2 | Contar los registros por tipo de taxi | `data/raw/*/2026/*.parquet` | 30,040,469 registros: 29,703,355 amarillos y 337,114 verdes | Los verdes son el 1.1 % de los viajes, por lo que las comparaciones entre tipos deben hacerse con proporciones o promedios y no con totales |
| Q3 | 3.3 y 3.4 | Identificar las columnas y sus tipos de datos | `data/raw/yellow/2026/*.parquet` y `data/raw/green/2026/*.parquet` | 21 columnas en amarillos y 22 en verdes, con 18 en común; `request_source` solo existe desde junio de 2026; fechas `TIMESTAMP`, montos `DOUBLE` y códigos categóricos `BIGINT` | Unificar las fechas con `COALESCE` y utilizar `union_by_name` en todas las lecturas, incluyendo `DESCRIBE`, para no perder columnas de archivos posteriores |
| Q4 | 3.4 | Comparar el tamaño en disco de columnas categóricas contra su tamaño en memoria | Metadatos de `data/raw/*/2026/*.parquet` | Entre 2.2 y 3.6 MB en disco contra cerca de 229 MB por columna en memoria | Leer únicamente las columnas que necesita cada consulta |
| Q5 | 3.5 | Obtener una muestra reproducible de registros | `data/raw/yellow/2026/*.parquet` y `data/raw/green/2026/*.parquet` | 5 registros por tipo con la semilla 123, de distintos meses | Se descartó `reservoir` por tomar solo enero, y se evitó ordenar todo el conjunto con todas sus columnas, ya que agotaba la memoria |
| Q6 | 3.6 | Crear una vista que unifique amarillos y verdes | `data/raw/*/2026/*.parquet` | Vista `viajes` con tipo, mes del archivo y fechas unificadas | Reutilizar esta estructura en los siguientes ejercicios |
| Q7 | 3.6 | Cuantificar los problemas de calidad de datos | Vista `viajes` | Conteos y porcentajes de cada problema por tipo de taxi | Los registros no se eliminan; se documentan para definir filtros en el Ejercicio 4 |
| Q8 | 3.6 | Verificar que los nulos aparecen en bloque | Vista `viajes` | El 26 % de amarillos y el 14.5 % de verdes tienen nulos simultáneos | Tratar este bloque como un grupo al analizar pasajeros, tarifas y formas de pago |
| Q9 | 3.6 | Identificar valores extremos | Vista `viajes` | Fechas desde 2001, distancias de hasta 328,522 millas y totales entre -2,560.2 y 7,053.5 | Definir rangos válidos de fechas, distancias y montos para el análisis |
| Q10 | 3.9 | Contar viajes por tipo de taxi en marzo de 2026 | `data/raw/*/2026/*.parquet` | 3,952,443 viajes amarillos y 44,203 verdes | Ejemplifica la lectura de únicamente las columnas necesarias |

## **3.9 Consultar directamente un archivo Parquet**

Consultar directamente un archivo Parquet significa que DuckDB lee el archivo tal como está almacenado en disco, sin tener que importarlo antes a una tabla ni cargarlo completo en memoria. Considero que esta estrategia es útil cuando el volumen de datos es grande porque permite enfocarse únicamente en la información que importa para cada consulta. Por ejemplo, si tuviera que hacer un informe de la cantidad de viajes de taxis amarillos contra verdes en un periodo específico, no me interesaría la tarifa calculada por tiempo y distancia, la propina ni los cargos adicionales, sino únicamente la fecha de inicio del viaje y el tipo de taxi. Dado que Parquet almacena los datos por columnas, DuckDB lee solamente esas columnas y omite el resto, de tal forma que no es necesario cargar los 30 millones de registros con todas sus columnas en memoria RAM.

Esto es especialmente importante en nuestro caso, ya que nuestras computadoras no cuentan con una gran cantidad de memoria. Si se cargara todo el conjunto de datos con pandas, probablemente nos quedaríamos sin memoria, tal como ocurrió en el 3.5 al intentar ordenar todos los registros con todas sus columnas.

In [11]:
import time

inicio = time.perf_counter()
viajes_marzo = con.sql(f"""
    SELECT regexp_extract(filename, '(yellow|green)_tripdata', 1) AS tipo,
           COUNT(*)                                              AS viajes
    FROM read_parquet('{RUTA_TODOS}', filename = true, union_by_name = true)
    WHERE COALESCE(tpep_pickup_datetime, lpep_pickup_datetime) >= TIMESTAMP '2026-03-01'
      AND COALESCE(tpep_pickup_datetime, lpep_pickup_datetime) <  TIMESTAMP '2026-04-01'
    GROUP BY tipo
    ORDER BY tipo DESC
""").df()
duracion = time.perf_counter() - inicio

print(f"Tiempo de la consulta: {duracion:.2f} s (solo se leen las columnas de fecha de inicio)")
viajes_marzo

Tiempo de la consulta: 0.63 s (solo se leen las columnas de fecha de inicio)


,tipo,viajes
0,yellow,3952443
1,green,44203
